# 深度學習框架應用（大學部）｜主題 11：注意力機制與 Transformer

這本筆記本配合投影片《主題 11：注意力機制與 Transformer》。五題都不用下載資料：前四題用小小的矩陣和亂數，第五題自己產生數字序列，CPU 幾秒到一分鐘就跑完。五題依序練：縮放點積注意力 → 兩種遮罩 → 多頭注意力 → 位置編碼 → 用 TransformerEncoder 訓練一個小任務。**練習一～五，每題 20%，合計 100%**。

| 練習 | 題目 | 投影片（練習頁） | 觀念在哪一頁 |
|---|---|---|---|
| 一 | 手刻縮放點積注意力 | 第 111 頁 | 第 27～48 頁 |
| 二 | 因果遮罩與 padding 遮罩 | 第 112 頁 | 第 79～85 頁 |
| 三 | 多頭注意力的形狀與參數量 | 第 113 頁 | 第 49～58 頁 |
| 四 | 正弦位置編碼 | 第 114 頁 | 第 59～69 頁 |
| 五 | 用 TransformerEncoder 把序列倒過來 | 第 115 頁 | 第 65、70～76、96～100 頁 |

**怎麼用**

1. 用私人 Google 帳號登入（不使用學校帳號），按上方的「複製到雲端硬碟」（或「檔案 › 在雲端硬碟中儲存複本」）另存一份自己的複本，之後都改自己那份。**沒有另存就直接寫，分頁一關就全部不見。** 存好後先把檔名改成 `深度學習_W11_學號_姓名.ipynb`（左上角點檔名就能改）。
2. 由上往下跑：先跑「環境準備」「共用小工具」兩格，之後每一題的「準備（直接執行）」那一格也要先跑。每一題只有一格標 ✏️ 的程式格：把每個 `___` 換成答案再執行，提示就寫在同一行後面的註解。還沒把 `___` 換掉就執行，會出現紅字錯誤（例如 TypeError、ValueError、AttributeError），錯誤訊息裡箭頭 `---->` 指的那一行就是還沒填的空格，填好再跑一次就好。
3. 不用 GPU，CPU 就夠了。練習五會訓練兩次小模型，在 GPU 上跑或 PyTorch 版本不同時，loss 的最後幾位數會和投影片不一樣，準確率和注意力圖的樣子相同就對了。改壞了就重新開原本的連結，再存一份複本。
4. **每題做完就截一張整個螢幕的圖（截圖 ①～⑤）**，要看得到身份指紋：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間，以及這一題的輸出。**看不到身份指紋，那一題不計分。**

**繳交**：做完照最後一格「繳交前檢查」，把**分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤）**交到 Flipclass 作業區。

## 環境準備

下面兩格五題都會用到：載入套件、設定印出來的小數位數，以及畫注意力熱力圖的 `show_heat`。熱力圖第 i 列是第 i 個字在「問」，第 j 欄是被看的第 j 個字，格子越亮，分到的注意力越多。

In [ ]:
# ===== 環境準備（直接執行）=====
import math, warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')                  # 不顯示版本提醒
torch.set_printoptions(precision=4, sci_mode=False)  # 印到小數 4 位
print('PyTorch：', torch.__version__)

In [ ]:
# ===== 共用小工具（直接執行，不用改）=====
def show_heat(w, words, title, cols=None):
    """把一張注意力權重表畫成熱力圖（圖上的字用英文，Colab 才不會變方塊）"""
    w = w.detach().cpu()
    plt.figure(figsize=(3.6, 3.2))
    plt.imshow(w, cmap='viridis', vmin=0, vmax=1)
    for i in range(w.size(0)):
        for j in range(w.size(1)):
            plt.text(j, i, f'{w[i, j]:.2f}', ha='center', va='center',
                     color='white' if w[i, j] < 0.6 else 'black', fontsize=8)
    plt.xticks(range(w.size(1)), cols or words); plt.yticks(range(w.size(0)), words)
    plt.xlabel('key (being looked at)'); plt.ylabel('query (asking)')
    plt.title(title); plt.colorbar(); plt.show()

## 練習一：手刻縮放點積注意力 (20%)

> 題目在投影片**第 111 頁**｜觀念在第 27～48 頁

**情境**：「我 吃 飯」三個字，每個字已經有自己的 q、k、v（d_k＝4，故意挑好算的數字）。每個字都要去「問」三個字（包含自己）：分數是 q 和 k 的內積，除以 √d_k，再用 softmax 變成加起來是 1 的權重，最後把三個字的 v 照權重加起來。

**這題在練**：把 Attention(Q, K, V)＝softmax(QKᵀ／√d_k)V 一行一行寫出來，並和 PyTorch 內建的 `F.scaled_dot_product_attention` 對答案

**步驟**

1. 先執行「練習一：準備」：準備好 Q、K、V 三張表（每一列是一個字）
2. 在 ✏️ 格填好三個空格：d_k 取哪一維、怎麼開根號、softmax 沿哪一軸
3. 先自己手算「我」那一列，再和印出來的分數、權重、輸出對一對；最後看一行「和 PyTorch 內建的一樣嗎？」

**要繳交**

- 截圖 ①：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習一：準備（直接執行，不用改）=====
words = ['I', 'eat', 'rice']                 # 「我 吃 飯」三個字
Q = torch.tensor([[1., 0, 1, 0],             # 我 的 q
                  [0., 1, 0, 1],             # 吃 的 q
                  [1., 1, 1, 0]])            # 飯 的 q
K = torch.tensor([[1., 1, 0, 0],             # 我 的 k
                  [0., 0, 1, 1],             # 吃 的 k
                  [1., 0, 1, 1]])            # 飯 的 k
V = torch.tensor([[1., 0],                   # 我 的 v
                  [0., 1],                   # 吃 的 v
                  [2., 2]])                  # 飯 的 v
print('Q', tuple(Q.shape), 'K', tuple(K.shape), 'V', tuple(V.shape))

In [ ]:
# ✏️ 練習一：手刻縮放點積注意力，再和 PyTorch 對答案
d_k = Q.size(___)            # ✏️ 最後一維
scores = Q @ K.transpose(-2, -1) / d_k ** ___  # ✏️ 開根號
w = scores.softmax(dim=___)  # ✏️ 沿 key
out = w @ V                  # 加權求和
print('分數 QKᵀ/√d_k：\n', scores)
print('權重：\n', w)
print('每一列加起來：', w.sum(-1))
print('輸出：\n', out)
ref = F.scaled_dot_product_attention(Q, K, V)
same = torch.allclose(out, ref, atol=1e-6)
print('和 PyTorch 內建的一樣嗎？', same)
print('不縮放時「我」那一列：', (Q @ K.T)[0].softmax(-1))
show_heat(w, words, 'scaled dot-product attention')

#### 📸 截圖 ①

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：分數、權重、每一列加起來、輸出、「和 PyTorch 內建的一樣嗎？ True」、不縮放的那一列，以及熱力圖

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 自己手算「我」那一列：分數 → 除以 √4 → 取 e 的次方 → 除以總和 → 加權求和，寫出每一步的數字，和印出來的對一對。**

（在這裡寫）

**2. 「吃」那一列的三個權重為什麼都是 0.3333？如果不除以 √d_k，「我」那一列會變成多少？哪一種分得比較「尖」？**

（在這裡寫）

## 練習二：因果遮罩與 padding 遮罩 (20%)

> 題目在投影片**第 112 頁**｜觀念在第 79～85 頁

**情境**：「我 想 吃 飯」的分數表已經算好（準備格的 S）。生成文字的時候，每個字只准看自己和左邊的字（因果遮罩）；一批句子長短不一，短句後面補的空格（padding）也不該被看。

**這題在練**：用 `torch.triu` 做出因果遮罩、用廣播做出 padding 遮罩，把不准看的分數換成 −∞；再認識 `F.scaled_dot_product_attention` 的布林遮罩方向

**步驟**

1. 先執行「練習二：準備」：S 是 4×4 的分數表，S2 是兩句話共用同一張表，sdpa 是內建的注意力函式
2. 在 ✏️ 格做出因果遮罩（對角線不遮），以及第 2 句後兩格的 padding 遮罩（補一軸才能廣播到每一列）
3. 最後用內建函式驗證：把遮罩翻過來（True＝看得到）傳給 attn_mask，結果要和 is_causal=True 一樣

**要繳交**

- 截圖 ②：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習二：準備（直接執行，不用改）=====
words = ['I', 'want', 'eat', 'rice']         # 「我 想 吃 飯」四個字
S = torch.tensor([[2., 1, 0, 1],             # 已經除過 √d_k 的分數表
                  [1., 2, 1, 0],             # 第 i 列＝第 i 個字在問
                  [0., 1, 2, 1],             # 第 j 欄＝被看的第 j 個字
                  [1., 0, 1, 2]])
S2 = S.expand(2, 4, 4)                        # 兩句話先共用同一張分數表
sdpa = F.scaled_dot_product_attention        # PyTorch 內建的注意力函式
q = torch.randn(1, 4, 8, generator=torch.Generator().manual_seed(0))
print('沒有遮罩時每一列加起來：', S.softmax(-1).sum(-1))

In [ ]:
# ✏️ 練習二：因果遮罩與 padding 遮罩
causal = torch.ones(4, 4).bool().triu(___)  # ✏️ 對角線
S_c = S.masked_fill(causal, float('___'))  # ✏️ 負無限大
w_c = S_c.softmax(dim=-1)
print('因果遮罩後的權重：\n', w_c)
# 第 2 句只有「我 想」兩個字，後 2 格是補的 padding
pad = torch.tensor([[0, 0, 0, 0], [0, 0, 1, 1]]).bool()
m = pad[:, ___, :]                    # ✏️ 補一軸 (2,1,4)
w_p = S2.masked_fill(m, float('-inf')).softmax(-1)
print('第 2 句的權重：\n', w_p[1])
a = sdpa(q, q, q, attn_mask=___)      # ✏️ True＝看得到
b = sdpa(q, q, q, is_causal=True)
print('一樣嗎？', torch.allclose(a, b, atol=1e-6))
show_heat(w_c, words, 'causal mask')

#### 📸 截圖 ②

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：因果遮罩後的權重（右上角都是 0）、第 2 句的權重（後兩欄都是 0）、「一樣嗎？ True」，以及熱力圖

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 因果遮罩後，「想」那一列為什麼只剩兩個不是 0 的數字？它們是怎麼算出來的？如果把 −inf 換成 0，結果會對嗎？**

（在這裡寫）

**2. 第 2 句的後兩格是 padding，權重變成什麼樣子？如果某一列「全部」都被遮掉會發生什麼事？`F.scaled_dot_product_attention` 的布林遮罩和 `nn.MultiheadAttention` 的有什麼不同？**

（在這裡寫）

## 練習三：多頭注意力的形狀與參數量 (20%)

> 題目在投影片**第 113 頁**｜觀念在第 49～58 頁

**情境**：同一批輸入 (2, 6, 128)：兩句話、每句 6 個字、每個字 128 維。分別用 1、8、16 頭的 `nn.MultiheadAttention` 跑一次，看輸出和權重的形狀、數一數參數；再把範例② 手刻的多頭注意力和官方版接上同一組權重，比一比輸出。

**這題在練**：確認頭數只改「切法」、不改參數量，並看懂分頭權重 (B, h, T, T) 的形狀

**步驟**

1. 先執行「練習三：準備」：裡面是範例① 的 scaled_dot_product、範例② 的 MultiHeadAttention，以及把權重複製到官方版的 copy_weights
2. 在 ✏️ 格填好第三種頭數、batch_first，以及數「一個張量裡有幾個數字」的方法（numel）
3. 比一比三行輸出的參數量，以及手刻版和官方版的最大差距

**要繳交**

- 截圖 ③：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習三：準備（直接執行，不用改）=====
MHA = nn.MultiheadAttention                  # 官方的多頭注意力（名字太長，取個小名）

def scaled_dot_product(q, k, v, mask=None):  # 範例①（投影片第 45 頁）
    d_k = q.size(-1)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn = F.softmax(scores, dim=-1)
    return attn @ v, attn

class MultiHeadAttention(nn.Module):        # 範例②（投影片第 54 頁）
    def __init__(self, d_model=128, n_head=8):
        super().__init__()
        self.h, self.d_k = n_head, d_model // n_head
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)
    def split(self, x):          # (B,T,D) -> (B,h,T,d_k)
        B, T, _ = x.shape
        return x.view(B, T, self.h, self.d_k).transpose(1, 2)
    def forward(self, x, mask=None):
        q, k = self.split(self.wq(x)), self.split(self.wk(x))
        v = self.split(self.wv(x))
        o, attn = scaled_dot_product(q, k, v, mask)
        B, _, T, _ = o.shape
        o = o.transpose(1, 2).reshape(B, T, self.h * self.d_k)
        return self.wo(o), attn

def copy_weights(mine, off):
    """官方版把 wq、wk、wv 疊成一個大矩陣 in_proj_weight，wo 叫 out_proj"""
    with torch.no_grad():
        off.in_proj_weight.copy_(torch.cat([mine.wq.weight, mine.wk.weight, mine.wv.weight]))
        off.in_proj_bias.copy_(torch.cat([mine.wq.bias, mine.wk.bias, mine.wv.bias]))
        off.out_proj.weight.copy_(mine.wo.weight)
        off.out_proj.bias.copy_(mine.wo.bias)

In [ ]:
# ✏️ 練習三：多頭注意力的形狀與參數量
torch.manual_seed(0)
x = torch.randn(2, 6, 128)             # (B, T, d_model)
for h in [1, 8, ___]:                  # ✏️ 16 頭
    m = MHA(128, h, batch_first=___)   # ✏️ 批次在前
    out, w = m(x, x, x, average_attn_weights=False)
    n = sum(p.___() for p in m.parameters())  # ✏️ 個數
    print(f'{h:2d} 頭｜out {tuple(out.shape)}｜'
          f'w {tuple(w.shape)}｜參數 {n:,}')
mine = MultiHeadAttention(128, 8)      # 範例②
off = MHA(128, 8, batch_first=True)    # 官方版
copy_weights(mine, off)                # 權重抄過去
diff = (mine(x)[0] - off(x, x, x)[0]).abs().max()
print(f'手刻版和官方版最大差距：{diff.item():.1e}')

#### 📸 截圖 ③

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：三行「頭數｜out｜w｜參數」，以及「手刻版和官方版最大差距」那一行

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 三種頭數的參數量各是多少？為什麼都一樣？寫出算式。**

（在這裡寫）

**2. `w` 的形狀 (2, 8, 6, 6) 四個數字各代表什麼？把 16 改成 6 再跑會出現什麼錯誤？為什麼？**

（在這裡寫）

## 練習四：正弦位置編碼 (20%)

> 題目在投影片**第 114 頁**｜觀念在第 59～69 頁

**情境**：注意力本身不分先後，所以要替每個位置做一個「位置條碼」加到字向量上。原論文用不同頻率的 sin、cos：偶數維用 sin、奇數維用 cos，越後面的維度轉得越慢。

**這題在練**：照公式 PE(pos, 2i)＝sin(pos／10000^(2i/d))、PE(pos, 2i＋1)＝cos(pos／10000^(2i/d)) 算出一張 50×16 的表，畫出來，並用餘弦相似度看「距離一樣，相似度就一樣」

**步驟**

1. 先執行「練習四：準備」：sim 算兩個位置有多像，show_pe 畫熱力圖和曲線
2. 在 ✏️ 格填好分母裡的維度，以及偶數維、奇數維的起點與函式
3. 看四行相似度：相鄰的兩對（10 和 11、30 和 31）一不一樣？離得越遠是不是越不像？

**要繳交**

- 截圖 ④：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習四：準備（直接執行，不用改）=====
def sim(pe, a, b):
    """位置 a 和位置 b 的編碼有多像（餘弦相似度：1＝方向一樣，0＝不相干）"""
    return F.cosine_similarity(pe[a], pe[b], dim=0).item()

def show_pe(pe):
    """左：整張位置編碼表的熱力圖；右：維度 0、4、8 隨位置怎麼變"""
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
    im = ax[0].imshow(pe, cmap='RdBu', aspect='auto', vmin=-1, vmax=1)
    ax[0].set_xlabel('dimension'); ax[0].set_ylabel('position')
    ax[0].set_title('sinusoidal positional encoding'); fig.colorbar(im, ax=ax[0])
    for d in (0, 4, 8):
        ax[1].plot(pe[:, d], label=f'dim {d}')
    ax[1].set_xlabel('position'); ax[1].legend(); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()

In [ ]:
# ✏️ 練習四：正弦位置編碼（50 個位置、16 維）
n_pos, d = 50, 16
pos = torch.arange(n_pos).float()[:, None]  # (50, 1)
i2 = torch.arange(0, d, 2).float()          # 0, 2, …, 14
ang = pos / 10000 ** (i2 / ___)             # ✏️ 除以維度
pe = torch.zeros(n_pos, d)
pe[:, 0::2] = torch.___(ang)                # ✏️ 偶數維
pe[:, ___::2] = torch.cos(ang)              # ✏️ 奇數維
print('位置 1 的前 4 維：', pe[1, :4])
for a, b in [(10, 11), (30, 31), (10, 20), (10, 40)]:
    print(f'位置 {a} 和 {b}：{sim(pe, a, b):.4f}')
show_pe(pe)                                 # 準備格寫好的

#### 📸 截圖 ④

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：位置 1 的前 4 維、四行相似度，以及熱力圖和曲線圖

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 位置 10 和 11、30 和 31 的相似度為什麼一模一樣？（提示：sin a sin b＋cos a cos b＝cos(a−b)）**

（在這裡寫）

**2. 熱力圖上，左邊（維度 0、1）和右邊（維度 14、15）哪一邊變化比較快？為什麼要混用快慢不同的頻率？**

（在這裡寫）

## 練習五：用 TransformerEncoder 把序列倒過來 (20%)

> 題目在投影片**第 115 頁**｜觀念在第 65、70～76、96～100 頁

**情境**：題目是 8 個 1～9 的數字，答案是同一串倒過來（例如 3 1 4 1 5 9 2 6 → 6 2 9 5 1 4 1 3）。模型在每個位置猜一個數字，所以第 0 個位置要去「抄」第 7 個位置的數字、第 1 個位置抄第 6 個…。準備格已經用「沒有位置編碼」練好一個對照組。

**這題在練**：用 `nn.TransformerEncoderLayer`、`nn.TransformerEncoder` 疊出編碼器，親眼看到位置編碼的作用，並從注意力圖看出模型「看了哪裡」

**步驟**

1. 先執行「練習五：準備＋對照組」：看對照組的準確率
2. 在 ✏️ 格填好頭數、batch_first，並打開位置編碼，訓練 400 步
3. 比一比兩個準確率，再看注意力熱力圖：每一列最亮的格子排成什麼形狀？

> 準備格和 ✏️ 格各訓練一次，CPU 上各大約 5～20 秒。

**要繳交**

- 截圖 ⑤：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習五：準備＋對照組（直接執行，不用改，約 10 秒）=====
T = 8                                        # 每一串 8 個數字

def make_batch(n):
    """n 串 1～9 的亂數；答案是同一串倒過來"""
    x = torch.randint(1, 10, (n, T))
    return x, x.flip(1)

class PosEnc(nn.Module):                     # 範例③（投影片第 65 頁）的正弦位置編碼
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class Reverser(nn.Module):
    """嵌入（＋位置編碼）→ 編碼器 → 每個位置猜一個數字（10 類）"""
    def __init__(self, encoder, use_pe=True, d=64):
        super().__init__()
        self.emb, self.pos = nn.Embedding(10, d), PosEnc(d)
        self.enc, self.out, self.use_pe = encoder, nn.Linear(d, 10), use_pe
    def embed(self, x):
        h = self.emb(x)
        return self.pos(h) if self.use_pe else h
    def forward(self, x):
        return self.out(self.enc(self.embed(x)))

def train(model, steps=400):
    """每一步拿 64 串新題目練一次，回傳每一步的 loss"""
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    model.train(); hist = []
    for _ in range(steps):
        x, y = make_batch(64)
        loss = F.cross_entropy(model(x).reshape(-1, 10), y.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        hist.append(loss.item())
    return hist

def accuracy(model):
    """拿 1,000 串沒看過的題目，算每個位置猜對的比例"""
    model.eval()
    with torch.no_grad():
        x, y = make_batch(1000)
        return (model(x).argmax(-1) == y).float().mean().item()

def attention_map(model, x):
    """取出第 1 層自注意力的權重（各頭平均），形狀 (T, T)"""
    model.eval()
    with torch.no_grad():
        h = model.embed(x)
        _, w = model.enc.layers[0].self_attn(h, h, h, need_weights=True)
    return w[0]

def show_attn(w, x, losses):
    """左：注意力熱力圖；右：有、沒有位置編碼的 loss 曲線"""
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    im = ax[0].imshow(w, cmap='viridis', vmin=0, vmax=1)
    ax[0].set_xticks(range(T)); ax[0].set_xticklabels(x[0].tolist())
    ax[0].set_yticks(range(T)); ax[0].set_yticklabels(x[0].tolist())
    ax[0].set_xlabel('key position'); ax[0].set_ylabel('query position')
    ax[0].set_title('layer 1 attention'); fig.colorbar(im, ax=ax[0])
    ax[1].plot(base_losses, label='no positional encoding')
    ax[1].plot(losses, label='with positional encoding')
    ax[1].set_xlabel('step'); ax[1].set_ylabel('loss'); ax[1].legend(); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()

torch.manual_seed(0)                         # 對照組：一樣的編碼器，只是不加位置編碼
lyr0 = nn.TransformerEncoderLayer(64, nhead=4, dim_feedforward=128,
                                  dropout=0.0, batch_first=True)
base = Reverser(nn.TransformerEncoder(lyr0, num_layers=1), use_pe=False)
base_losses = train(base)
print(f'對照組（沒有位置編碼）：準確率 {accuracy(base):.3f}')
demo = torch.tensor([[3, 1, 4, 1, 5, 9, 2, 6]])   # 等一下拿來看注意力的那一串
print('題目：', demo[0].tolist(), '→ 答案：', demo.flip(1)[0].tolist())

In [ ]:
# ✏️ 練習五：加上位置編碼，學會把序列倒過來
torch.manual_seed(0)
lyr = nn.TransformerEncoderLayer(64, nhead=___,  # ✏️ 4 頭
          dim_feedforward=128, dropout=0.0,
          batch_first=___)          # ✏️ 批次在前
enc = nn.TransformerEncoder(lyr, num_layers=1)  # 疊 1 層
model = Reverser(enc, use_pe=___)   # ✏️ 要位置編碼
losses = train(model)               # 練 400 步
print(f'有位置編碼：準確率 {accuracy(model):.3f}')
w = attention_map(model, demo)      # (8, 8)
print('每一列最亮在第幾欄：', w.argmax(-1).tolist())
show_attn(w, demo, losses)          # 熱力圖＋兩條 loss

#### 📸 截圖 ⑤

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：對照組的準確率、「有位置編碼：準確率」那一行、「每一列最亮在第幾欄」那一行，以及熱力圖和兩條 loss 曲線

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 對照組（沒有位置編碼）的準確率大約多少？為什麼它學不會把序列倒過來？**

（在這裡寫）

**2. 注意力圖上每一列最亮的格子排成什麼形狀？第 0 個位置主要在看第幾個位置？為什麼？**

（在這裡寫）

## AI 使用紀錄

有用 AI 就寫下來，沒用就寫「沒有使用」。

| 用了哪個 AI | 問了什麼（大概就好） | 它哪裡幫到你 | 它哪裡說錯、你怎麼發現的 |
|---|---|---|---|
| （例：ChatGPT） | （例：為什麼注意力要除以根號 d_k） |  |  |

## 繳交前檢查

1. **重跑一次**：「執行階段 › 重新啟動工作階段並執行所有儲存格」，每一格都有輸出、沒有紅字。
2. **檔名**：`深度學習_W11_學號_姓名.ipynb`（左上角點檔名就能改）。
3. **截圖**：5 張（①～⑤），每張都是整個螢幕，看得到身份指紋（左上的檔名、右上的 Google 頭貼、右下工作列的日期時間）和那一題的輸出。
4. **開放分享**：右上角「分享」→「一般存取權」改成「知道連結的任何人」→ 角色選「檢視者」→「複製連結」。
5. **自己檢查**：開一個無痕視窗貼上連結，不用登入就打得開才算成功。
6. **下載**：「檔案 › 下載 › 下載 .ipynb」。
7. **交到 Flipclass**：分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤），交到 Flipclass 作業區，期限照 Flipclass 規定。